# HGCTM Revision — Post Hoc Topic Validation Against Known Deposit Types

This notebook addresses **Reviewer 1, Comment 1** and strengthens the responses to **Reviewer 2, Comments 1, 2, and 7**.

The final HGCTM-S model was fitted without using the five database deposit-type labels. The labels are therefore used here only as an external post hoc validation target.

## Scientific question

Do the final seven assemblage modes show statistically significant and geologically interpretable enrichment across known copper-deposit types?

## Validation strategy

The analysis uses two continuous quantities:

1. **Posterior attribution weights** used in the submitted workflow  
   \[
   r_{ik} \propto 	heta_{ik}
   \prod_f \phi_{ikf}^{x_{if}}
   \]
   These weights combine covariate-derived prevalence, context-conditioned topic composition, and the observed mineral-family record.

2. **Topic prevalence** \(	heta_{ik}\)  
   This shows the prevalence pattern implied by lithology and age alone.

The posterior attribution weights are the primary label-validation quantity. Topic prevalence is reported separately so that mineral-content evidence is not confused with covariate-driven prevalence.

For every topic, the notebook reports:

- mean and median attribution by deposit type;
- Kruskal–Wallis tests and epsilon-squared effect sizes;
- pairwise Mann–Whitney tests with Holm correction;
- enrichment ratios with stratified bootstrap confidence intervals;
- label-permutation tests, including max-statistic adjusted cell-level \(p\)-values;
- dominant-attribution contingency, chi-square test, Cramér's \(V\), and adjusted residuals;
- pre-specified checks for the four modes expected to be enriched in a particular type:
  - T1 ↔ Porphyry;
  - T4 ↔ Magmatic sulphide;
  - T5 ↔ IOCG;
  - T6 ↔ VMS.

The analysis is repeated for:

- the complete final-model cohort of 1,335 deposits;
- the coordinate-valid cohort of 1,237 deposits.

No classifier is trained, no deposit-type label is used for model fitting, and no topic is required to correspond one-to-one with a deposit class.

## Required Kaggle inputs

Attach the following existing inputs:

### Required

1. **The original final-model results bundle** produced by the main `gcdd-hgctm-s.ipynb` notebook.  
   It may be attached either as a folder or as `hgctm_results_bundle.zip`. It must contain:

   - `theta_m7b.csv`
   - `phi_m7b_global.csv`
   - `mu_m7b.npy`
   - `delta_litho_m7b.npy`
   - `delta_age_m7b.npy`
   - `tau_m7b.npy`
   - `scalars_m7b.json`
   - `covariates_model_set.csv`

2. `Hgctm_stage0.zip`, containing `X_family_copper.csv`.

### Not required

The newer prior, lithology-source, presence/absence, documentation-depth, and geographic-audit outputs are not needed for this post hoc validation. They remain separate robustness analyses.

If the original results were added in Kaggle as the dataset named `hgctm-results-bandle`, the notebook will locate that folder automatically.

In [ ]:
# Install only missing packages.

import importlib.util
import subprocess
import sys

required = {
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
from scipy.stats import (
    chi2_contingency,
    kruskal,
    mannwhitneyu,
)

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_topic_validation_deposit_types"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"

for folder in [OUT, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------- SETTINGS ----------------

BOOTSTRAP_REPLICATES = 2000
PERMUTATION_REPLICATES = 5000
RANDOM_SEED = 20260622

EXPECTED_N = 1335
EXPECTED_VALID_N = 1237
EXPECTED_F = 35
EXPECTED_K = 7

TYPE_ORDER = [
    "Porphyry",
    "VMS",
    "Sediment-Hosted",
    "Magmatic sulfide",
    "IOCG",
]

TOPIC_LABELS = [
    "T1 Porphyry-enriched mixed alteration",
    "T2 Supergene Cu-enrichment overprint",
    "T3 Broad polymetallic sulphide background",
    "T4 Magmatic-sulphide-enriched Ni-Co-As",
    "T5 Sodic-calcic silicate-halide mode",
    "T6 VMS-enriched Pb-Zn-Cu-barite",
    "T7 Diffuse sulphate-phosphate residual",
]

EXPECTED_TYPE_BY_TOPIC = {
    1: "Porphyry",
    4: "Magmatic sulfide",
    5: "IOCG",
    6: "VMS",
}

print("Bootstrap replicates:", BOOTSTRAP_REPLICATES)
print("Permutation replicates:", PERMUTATION_REPLICATES)


## 1. Locate and extract the frozen inputs

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)

    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(
        datetime.now(timezone.utc).isoformat(),
        encoding="utf-8",
    )

    return target


def find_folder_with(required_names, label):
    required_names = set(required_names)
    roots = [Path("/kaggle/input"), WORK]

    # Search existing folders first.
    for root in roots:
        if not root.exists():
            continue

        for first_name in required_names:
            for path in root.rglob(first_name):
                folder = path.parent
                available = {
                    item.name
                    for item in folder.iterdir()
                    if item.is_file()
                }

                if required_names.issubset(available):
                    return folder

    # Search ZIP archives.
    for root in roots:
        if not root.exists():
            continue

        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }

                    if required_names.issubset(basenames):
                        safe_label = re.sub(
                            r"[^a-z0-9]+",
                            "_",
                            label.lower(),
                        ).strip("_")

                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:10]

                        target = (
                            WORK
                            / f"{safe_label}_{digest}"
                        )

                        extract_zip_once(
                            zip_path,
                            target,
                        )

                        for first_name in required_names:
                            for match in target.rglob(first_name):
                                folder = match.parent
                                available = {
                                    item.name
                                    for item in folder.iterdir()
                                    if item.is_file()
                                }

                                if required_names.issubset(available):
                                    return folder

            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        f"Could not locate {label}. Required files: "
        f"{sorted(required_names)}"
    )


FINAL_RESULTS_REQUIRED = {
    "theta_m7b.csv",
    "phi_m7b_global.csv",
    "mu_m7b.npy",
    "delta_litho_m7b.npy",
    "delta_age_m7b.npy",
    "tau_m7b.npy",
    "scalars_m7b.json",
    "covariates_model_set.csv",
}

FINAL_RESULTS = find_folder_with(
    FINAL_RESULTS_REQUIRED,
    "original final HGCTM results",
)

STAGE0 = find_folder_with(
    {"X_family_copper.csv"},
    "HGCTM Stage 0",
)

print("Final-model results:", FINAL_RESULTS)
print("Stage 0:", STAGE0)


## 2. Load and audit the original final HGCTM-S solution

In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)

        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def canonicalize_index(frame, label):
    frame = frame.copy()
    frame.index = [
        canonical_id(value)
        for value in frame.index
    ]
    frame.index.name = "Mindat_id"

    if frame.index.isna().any():
        raise ValueError(
            f"{label} contains missing IDs."
        )

    if frame.index.duplicated().any():
        duplicates = frame.index[
            frame.index.duplicated()
        ].tolist()[:10]

        raise ValueError(
            f"{label} contains duplicate IDs: {duplicates}"
        )

    return frame


with open(
    FINAL_RESULTS / "scalars_m7b.json",
    "r",
    encoding="utf-8",
) as handle:
    scalars = json.load(handle)

K = int(scalars["K"])
F = int(scalars["F"])

if K != EXPECTED_K or F != EXPECTED_F:
    raise ValueError(
        f"Expected K={EXPECTED_K}, F={EXPECTED_F}; "
        f"found K={K}, F={F}."
    )

families = [
    str(value)
    for value in scalars[
        "families_fc"
    ]
]

lithology_levels = [
    str(value)
    for value in scalars[
        "litho_classes"
    ]
]

age_levels = [
    str(value)
    for value in scalars[
        "age_classes"
    ]
]

omega_a = float(
    scalars["omega_a"]
)

theta = canonicalize_index(
    pd.read_csv(
        FINAL_RESULTS / "theta_m7b.csv",
        index_col=0,
    ),
    "theta_m7b",
)

covariates = canonicalize_index(
    pd.read_csv(
        FINAL_RESULTS
        / "covariates_model_set.csv",
        index_col=0,
    ),
    "covariates_model_set",
)

X_raw = canonicalize_index(
    pd.read_csv(
        STAGE0 / "X_family_copper.csv",
        index_col=0,
    ),
    "X_family_copper",
)

phi_global = pd.read_csv(
    FINAL_RESULTS
    / "phi_m7b_global.csv"
).to_numpy(dtype=float)

mu = np.load(
    FINAL_RESULTS
    / "mu_m7b.npy"
)

delta_litho = np.load(
    FINAL_RESULTS
    / "delta_litho_m7b.npy"
)

delta_age = np.load(
    FINAL_RESULTS
    / "delta_age_m7b.npy"
)

tau = np.load(
    FINAL_RESULTS
    / "tau_m7b.npy"
)

# Reconstruct the exact final-model family matrix.
model_mask = (
    X_raw.sum(axis=1) >= 3
)

X_pre_family = (
    X_raw.loc[
        model_mask
    ].copy()
)

family_presence = (
    (X_pre_family > 0)
    .sum(axis=0)
)

family_keep = family_presence[
    family_presence >= 10
].index.tolist()

X_model = (
    X_pre_family[
        family_keep
    ].copy()
)

missing_families = (
    set(families)
    - set(X_model.columns)
)

if missing_families:
    raise ValueError(
        "Final-model families missing from Stage 0: "
        f"{sorted(missing_families)}"
    )

X_model = X_model[
    families
]

common_ids = [
    mindat_id
    for mindat_id in theta.index
    if (
        mindat_id in covariates.index
        and mindat_id in X_model.index
    )
]

theta = theta.loc[
    common_ids
].copy()

covariates = covariates.loc[
    common_ids
].copy()

X_model = X_model.loc[
    common_ids
].copy()

if len(common_ids) != EXPECTED_N:
    raise ValueError(
        f"Expected {EXPECTED_N} aligned deposits; "
        f"found {len(common_ids)}."
    )

if theta.shape != (
    EXPECTED_N,
    EXPECTED_K,
):
    raise ValueError(
        f"Unexpected theta shape: {theta.shape}"
    )

if X_model.shape != (
    EXPECTED_N,
    EXPECTED_F,
):
    raise ValueError(
        f"Unexpected X shape: {X_model.shape}"
    )

parameter_shapes = {
    "phi_global": phi_global.shape,
    "mu": mu.shape,
    "delta_litho": delta_litho.shape,
    "delta_age": delta_age.shape,
    "tau": tau.shape,
}

print("Parameter shapes:", parameter_shapes)

if not np.allclose(
    theta.sum(axis=1).to_numpy(),
    1.0,
    atol=1e-5,
):
    raise ValueError(
        "Theta rows do not sum to one."
    )

if not np.allclose(
    phi_global.sum(axis=1),
    1.0,
    atol=1e-5,
):
    raise ValueError(
        "Global topic rows do not sum to one."
    )

print("Aligned deposits:", len(common_ids))
print("Mineral families:", X_model.shape[1])
print("Deposit-type counts:")
print(
    covariates[
        "Deposit_type"
    ].value_counts().to_string()
)


## 3. Reconstruct context-conditioned topic compositions and posterior attribution weights

The submitted workflow used normalized attribution weights

\[
r_{ik} \propto 	heta_{ik}
\exp\left(
\sum_f x_{if}\log\phi_{ikf}
ight).
\]

These weights are a post hoc attribution diagnostic for the fitted mixed-membership model. They should not be described as a new supervised prediction or as a one-to-one genetic classification.

In [ ]:
lithology_to_index = {
    label: index
    for index, label
    in enumerate(
        lithology_levels
    )
}

age_to_index = {
    label: index
    for index, label
    in enumerate(
        age_levels
    )
}

lithology_values = (
    covariates[
        "litho_class"
    ]
    .fillna("unknown")
    .astype(str)
    .str.strip()
)

age_values = (
    covariates[
        "age_category"
    ]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

unknown_lithology_labels = sorted(
    set(lithology_values)
    - set(lithology_to_index)
)

unknown_age_labels = sorted(
    set(age_values)
    - set(age_to_index)
)

if unknown_lithology_labels:
    raise ValueError(
        "Unmapped lithology labels: "
        f"{unknown_lithology_labels}"
    )

if unknown_age_labels:
    raise ValueError(
        "Unmapped age labels: "
        f"{unknown_age_labels}"
    )

lithology_idx = (
    lithology_values
    .map(
        lithology_to_index
    )
    .to_numpy(
        dtype=int
    )
)

age_idx = (
    age_values
    .map(
        age_to_index
    )
    .to_numpy(
        dtype=int
    )
)

# Context-conditioned topic-family probabilities.
lithology_deviation = (
    delta_litho[
        :,
        lithology_idx,
        :,
    ].transpose(
        1,
        0,
        2,
    )
)

age_deviation = (
    delta_age[
        :,
        age_idx,
        :,
    ].transpose(
        1,
        0,
        2,
    )
)

psi = (
    mu[None, :, :]
    + tau[None, None, :]
    * lithology_deviation
    + omega_a
    * tau[None, None, :]
    * age_deviation
)

psi = (
    psi
    - psi.max(
        axis=2,
        keepdims=True,
    )
)

phi_context = np.exp(psi)
phi_context /= phi_context.sum(
    axis=2,
    keepdims=True,
)

if not np.allclose(
    phi_context.sum(axis=2),
    1.0,
    atol=1e-5,
):
    raise ValueError(
        "Context-conditioned topic rows do not sum to one."
    )

theta_np = theta.to_numpy(
    dtype=float
)

X_np = X_model.to_numpy(
    dtype=float
)

log_theta = np.log(
    np.clip(
        theta_np,
        1e-12,
        None,
    )
)

log_phi = np.log(
    np.clip(
        phi_context,
        1e-12,
        None,
    )
)

log_likelihood_by_topic = np.einsum(
    "nf,nkf->nk",
    X_np,
    log_phi,
)

log_attribution = (
    log_theta
    + log_likelihood_by_topic
)

log_attribution -= logsumexp(
    log_attribution,
    axis=1,
    keepdims=True,
)

responsibility = np.exp(
    log_attribution
)

dominant_topic = np.argmax(
    responsibility,
    axis=1,
)

if not np.allclose(
    responsibility.sum(axis=1),
    1.0,
    atol=1e-6,
):
    raise ValueError(
        "Attribution rows do not sum to one."
    )

responsibility_df = pd.DataFrame(
    responsibility,
    index=common_ids,
    columns=[
        f"T{topic}"
        for topic in range(
            1,
            K + 1,
        )
    ],
)

responsibility_df.to_csv(
    TABLE_DIR
    / "posterior_attribution_weights_full_1335.csv"
)

print("Posterior-attribution matrix:", responsibility.shape)
print("Dominant-attribution counts:")
print(
    pd.Series(
        dominant_topic + 1
    ).value_counts().sort_index().to_string()
)


## 4. Define the two validation cohorts

In [ ]:
def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False

    try:
        lat = float(lat)
        lon = float(lon)
    except Exception:
        return False

    if not (
        -90 <= lat <= 90
        and -180 <= lon <= 180
    ):
        return False

    if (
        abs(lat) < 1e-12
        and abs(lon) < 1e-12
    ):
        return False

    return True


coordinate_valid = covariates.apply(
    lambda row: valid_coordinate(
        row["Latitude"],
        row["Longitude"],
    ),
    axis=1,
).to_numpy()

if int(
    coordinate_valid.sum()
) != EXPECTED_VALID_N:
    raise ValueError(
        f"Expected {EXPECTED_VALID_N} coordinate-valid deposits; "
        f"found {int(coordinate_valid.sum())}."
    )

cohorts = {
    "full_1335": np.ones(
        EXPECTED_N,
        dtype=bool,
    ),
    "coordinate_valid_1237": (
        coordinate_valid
    ),
}

cohort_audit_rows = []

for cohort_name, mask in cohorts.items():
    frame = covariates.loc[
        np.array(common_ids)[mask]
    ]

    for deposit_type in TYPE_ORDER:
        cohort_audit_rows.append({
            "cohort": cohort_name,
            "deposit_type": deposit_type,
            "count": int(
                frame[
                    "Deposit_type"
                ].eq(
                    deposit_type
                ).sum()
            ),
        })

cohort_audit = pd.DataFrame(
    cohort_audit_rows
)

cohort_audit.to_csv(
    TABLE_DIR
    / "cohort_deposit_type_counts.csv",
    index=False,
)

print(
    cohort_audit.pivot(
        index="deposit_type",
        columns="cohort",
        values="count",
    ).to_string()
)


## 5. Statistical utilities

In [ ]:
def benjamini_hochberg(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(p_values)
    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted_ranked = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
        )
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[
            ::-1
        ]
    )[::-1]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0,
        1,
    )

    adjusted = np.empty(
        n,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_ranked

    return adjusted


def holm_adjust(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(p_values)
    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted_ranked = np.maximum.accumulate(
        (
            n
            - np.arange(n)
        )
        * ranked
    )

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0,
        1,
    )

    adjusted = np.empty(
        n,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_ranked

    return adjusted


def epsilon_squared(
    h_statistic,
    n,
    groups,
):
    denominator = (
        n - groups
    )

    if denominator <= 0:
        return np.nan

    value = (
        h_statistic
        - groups
        + 1
    ) / denominator

    return float(
        max(
            0.0,
            value,
        )
    )


def cramers_v(
    contingency,
):
    contingency = np.asarray(
        contingency,
        dtype=float,
    )

    chi_square, _, _, _ = (
        chi2_contingency(
            contingency,
            correction=False,
        )
    )

    n = contingency.sum()
    rows, columns = (
        contingency.shape
    )

    denominator = min(
        rows - 1,
        columns - 1,
    )

    if (
        n <= 0
        or denominator <= 0
    ):
        return np.nan

    return float(
        np.sqrt(
            (
                chi_square
                / n
            )
            / denominator
        )
    )


def adjusted_standardized_residuals(
    observed,
):
    observed = np.asarray(
        observed,
        dtype=float,
    )

    row_totals = observed.sum(
        axis=1,
        keepdims=True,
    )

    column_totals = observed.sum(
        axis=0,
        keepdims=True,
    )

    total = observed.sum()

    expected = (
        row_totals
        @ column_totals
        / total
    )

    row_proportions = (
        row_totals
        / total
    )

    column_proportions = (
        column_totals
        / total
    )

    denominator = np.sqrt(
        expected
        * (
            1
            - row_proportions
        )
        * (
            1
            - column_proportions
        )
    )

    residuals = (
        observed
        - expected
    ) / np.clip(
        denominator,
        1e-12,
        None,
    )

    return residuals


def format_significance(
    p_value,
):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


## 6. Cohort-level validation function

In [ ]:
def analyse_cohort(
    cohort_name,
    mask,
):
    rng = np.random.default_rng(
        RANDOM_SEED
        + (
            0
            if cohort_name
            == "full_1335"
            else 100
        )
    )

    labels = (
        covariates.loc[
            np.array(common_ids)[mask],
            "Deposit_type",
        ]
        .astype(str)
        .to_numpy()
    )

    r_values = responsibility[
        mask
    ]

    theta_values = theta_np[
        mask
    ]

    dominant_values = (
        dominant_topic[
            mask
        ]
    )

    n = len(labels)

    type_indices = {
        deposit_type: np.flatnonzero(
            labels
            == deposit_type
        )
        for deposit_type
        in TYPE_ORDER
    }

    # ---------------------------------------------------------
    # Descriptive matrices
    # ---------------------------------------------------------

    mean_r = np.zeros(
        (
            len(TYPE_ORDER),
            K,
        )
    )

    median_r = np.zeros_like(
        mean_r
    )

    mean_theta = np.zeros_like(
        mean_r
    )

    median_theta = np.zeros_like(
        mean_r
    )

    for type_index, deposit_type in enumerate(
        TYPE_ORDER
    ):
        indices = type_indices[
            deposit_type
        ]

        mean_r[
            type_index
        ] = r_values[
            indices
        ].mean(axis=0)

        median_r[
            type_index
        ] = np.median(
            r_values[
                indices
            ],
            axis=0,
        )

        mean_theta[
            type_index
        ] = theta_values[
            indices
        ].mean(axis=0)

        median_theta[
            type_index
        ] = np.median(
            theta_values[
                indices
            ],
            axis=0,
        )

    topic_columns = [
        f"T{topic}"
        for topic in range(
            1,
            K + 1,
        )
    ]

    mean_r_df = pd.DataFrame(
        mean_r,
        index=TYPE_ORDER,
        columns=topic_columns,
    )

    median_r_df = pd.DataFrame(
        median_r,
        index=TYPE_ORDER,
        columns=topic_columns,
    )

    mean_theta_df = pd.DataFrame(
        mean_theta,
        index=TYPE_ORDER,
        columns=topic_columns,
    )

    median_theta_df = pd.DataFrame(
        median_theta,
        index=TYPE_ORDER,
        columns=topic_columns,
    )

    mean_r_df.to_csv(
        TABLE_DIR
        / f"{cohort_name}__mean_posterior_attribution_by_type.csv"
    )

    median_r_df.to_csv(
        TABLE_DIR
        / f"{cohort_name}__median_posterior_attribution_by_type.csv"
    )

    mean_theta_df.to_csv(
        TABLE_DIR
        / f"{cohort_name}__mean_theta_by_type.csv"
    )

    median_theta_df.to_csv(
        TABLE_DIR
        / f"{cohort_name}__median_theta_by_type.csv"
    )

    # ---------------------------------------------------------
    # Kruskal-Wallis tests
    # ---------------------------------------------------------

    kw_rows = []

    for topic_index in range(K):
        groups = [
            r_values[
                type_indices[
                    deposit_type
                ],
                topic_index,
            ]
            for deposit_type
            in TYPE_ORDER
        ]

        h_statistic, p_value = (
            kruskal(
                *groups,
                nan_policy="omit",
            )
        )

        kw_rows.append({
            "cohort": cohort_name,
            "topic": (
                topic_index + 1
            ),
            "topic_label": (
                TOPIC_LABELS[
                    topic_index
                ]
            ),
            "n": int(n),
            "groups": int(
                len(TYPE_ORDER)
            ),
            "kruskal_H": float(
                h_statistic
            ),
            "p_value": float(
                p_value
            ),
            "epsilon_squared": (
                epsilon_squared(
                    h_statistic,
                    n,
                    len(TYPE_ORDER),
                )
            ),
        })

    kw_table = pd.DataFrame(
        kw_rows
    )

    kw_table[
        "p_fdr_bh"
    ] = benjamini_hochberg(
        kw_table[
            "p_value"
        ].to_numpy()
    )

    kw_table.to_csv(
        TABLE_DIR
        / f"{cohort_name}__kruskal_wallis_topics.csv",
        index=False,
    )

    # ---------------------------------------------------------
    # Pairwise Mann-Whitney tests
    # ---------------------------------------------------------

    pairwise_rows = []

    for topic_index in range(K):
        topic_rows = []

        for first_index in range(
            len(TYPE_ORDER)
        ):
            for second_index in range(
                first_index + 1,
                len(TYPE_ORDER),
            ):
                first_type = (
                    TYPE_ORDER[
                        first_index
                    ]
                )

                second_type = (
                    TYPE_ORDER[
                        second_index
                    ]
                )

                first_values = r_values[
                    type_indices[
                        first_type
                    ],
                    topic_index,
                ]

                second_values = r_values[
                    type_indices[
                        second_type
                    ],
                    topic_index,
                ]

                result = mannwhitneyu(
                    first_values,
                    second_values,
                    alternative="two-sided",
                    method="asymptotic",
                )

                rank_biserial = (
                    2
                    * result.statistic
                    / (
                        len(first_values)
                        * len(
                            second_values
                        )
                    )
                    - 1
                )

                topic_rows.append({
                    "cohort": (
                        cohort_name
                    ),
                    "topic": (
                        topic_index + 1
                    ),
                    "topic_label": (
                        TOPIC_LABELS[
                            topic_index
                        ]
                    ),
                    "type_1": first_type,
                    "type_2": second_type,
                    "n_1": int(
                        len(first_values)
                    ),
                    "n_2": int(
                        len(
                            second_values
                        )
                    ),
                    "mean_1": float(
                        first_values.mean()
                    ),
                    "mean_2": float(
                        second_values.mean()
                    ),
                    "median_1": float(
                        np.median(
                            first_values
                        )
                    ),
                    "median_2": float(
                        np.median(
                            second_values
                        )
                    ),
                    "U": float(
                        result.statistic
                    ),
                    "p_value": float(
                        result.pvalue
                    ),
                    "rank_biserial_type1_vs_type2": float(
                        rank_biserial
                    ),
                })

        topic_p_values = [
            row[
                "p_value"
            ]
            for row
            in topic_rows
        ]

        topic_adjusted = (
            holm_adjust(
                topic_p_values
            )
        )

        for row, adjusted in zip(
            topic_rows,
            topic_adjusted,
        ):
            row[
                "p_holm_within_topic"
            ] = float(
                adjusted
            )

        pairwise_rows.extend(
            topic_rows
        )

    pairwise_table = pd.DataFrame(
        pairwise_rows
    )

    pairwise_table[
        "p_fdr_bh_all_pairs"
    ] = benjamini_hochberg(
        pairwise_table[
            "p_value"
        ].to_numpy()
    )

    pairwise_table.to_csv(
        TABLE_DIR
        / f"{cohort_name}__pairwise_mannwhitney_holm.csv",
        index=False,
    )

    # ---------------------------------------------------------
    # Bootstrap enrichment ratios
    # ---------------------------------------------------------

    bootstrap_type_means = {}

    for deposit_type in TYPE_ORDER:
        indices = type_indices[
            deposit_type
        ]

        sampled_indices = rng.choice(
            indices,
            size=(
                BOOTSTRAP_REPLICATES,
                len(indices),
            ),
            replace=True,
        )

        bootstrap_type_means[
            deposit_type
        ] = r_values[
            sampled_indices
        ].mean(axis=1)

    bootstrap_overall_mean = sum(
        len(
            type_indices[
                deposit_type
            ]
        )
        * bootstrap_type_means[
            deposit_type
        ]
        for deposit_type
        in TYPE_ORDER
    ) / n

    overall_mean = r_values.mean(
        axis=0
    )

    enrichment_rows = []

    for type_index, deposit_type in enumerate(
        TYPE_ORDER
    ):
        ratios = (
            bootstrap_type_means[
                deposit_type
            ]
            / np.clip(
                bootstrap_overall_mean,
                1e-12,
                None,
            )
        )

        differences = (
            bootstrap_type_means[
                deposit_type
            ]
            - bootstrap_overall_mean
        )

        for topic_index in range(K):
            observed_ratio = (
                mean_r[
                    type_index,
                    topic_index,
                ]
                / max(
                    overall_mean[
                        topic_index
                    ],
                    1e-12,
                )
            )

            observed_difference = (
                mean_r[
                    type_index,
                    topic_index,
                ]
                - overall_mean[
                    topic_index
                ]
            )

            enrichment_rows.append({
                "cohort": cohort_name,
                "deposit_type": (
                    deposit_type
                ),
                "topic": (
                    topic_index + 1
                ),
                "topic_label": (
                    TOPIC_LABELS[
                        topic_index
                    ]
                ),
                "type_mean": float(
                    mean_r[
                        type_index,
                        topic_index,
                    ]
                ),
                "overall_mean": float(
                    overall_mean[
                        topic_index
                    ]
                ),
                "enrichment_ratio": float(
                    observed_ratio
                ),
                "ratio_ci_low": float(
                    np.quantile(
                        ratios[
                            :,
                            topic_index,
                        ],
                        0.025,
                    )
                ),
                "ratio_ci_high": float(
                    np.quantile(
                        ratios[
                            :,
                            topic_index,
                        ],
                        0.975,
                    )
                ),
                "mean_difference": float(
                    observed_difference
                ),
                "difference_ci_low": float(
                    np.quantile(
                        differences[
                            :,
                            topic_index,
                        ],
                        0.025,
                    )
                ),
                "difference_ci_high": float(
                    np.quantile(
                        differences[
                            :,
                            topic_index,
                        ],
                        0.975,
                    )
                ),
            })

    enrichment_table = pd.DataFrame(
        enrichment_rows
    )

    # ---------------------------------------------------------
    # Label permutation tests
    # ---------------------------------------------------------

    observed_group_means = (
        mean_r.copy()
    )

    observed_deviation = (
        observed_group_means
        - overall_mean[
            None,
            :
        ]
    )

    observed_between_group = np.sum(
        np.array([
            len(
                type_indices[
                    deposit_type
                ]
            )
            for deposit_type
            in TYPE_ORDER
        ])[
            :,
            None,
        ]
        * observed_deviation ** 2,
        axis=0,
    )

    cell_exceed = np.zeros(
        (
            len(TYPE_ORDER),
            K,
        ),
        dtype=int,
    )

    max_cell_exceed = np.zeros_like(
        cell_exceed
    )

    global_exceed = np.zeros(
        K,
        dtype=int,
    )

    observed_abs_deviation = np.abs(
        observed_deviation
    )

    for _ in range(
        PERMUTATION_REPLICATES
    ):
        permuted_labels = rng.permutation(
            labels
        )

        permuted_means = np.zeros_like(
            observed_group_means
        )

        permuted_counts = np.zeros(
            len(TYPE_ORDER),
            dtype=int,
        )

        for type_index, deposit_type in enumerate(
            TYPE_ORDER
        ):
            indices = np.flatnonzero(
                permuted_labels
                == deposit_type
            )

            permuted_counts[
                type_index
            ] = len(
                indices
            )

            permuted_means[
                type_index
            ] = r_values[
                indices
            ].mean(axis=0)

        permuted_deviation = (
            permuted_means
            - overall_mean[
                None,
                :
            ]
        )

        permuted_between_group = np.sum(
            permuted_counts[
                :,
                None,
            ]
            * permuted_deviation ** 2,
            axis=0,
        )

        global_exceed += (
            permuted_between_group
            >= observed_between_group
        )

        abs_permuted = np.abs(
            permuted_deviation
        )

        cell_exceed += (
            abs_permuted
            >= observed_abs_deviation
        )

        max_abs = abs_permuted.max()

        max_cell_exceed += (
            max_abs
            >= observed_abs_deviation
        )

    global_permutation_rows = []

    for topic_index in range(K):
        global_permutation_rows.append({
            "cohort": cohort_name,
            "topic": (
                topic_index + 1
            ),
            "topic_label": (
                TOPIC_LABELS[
                    topic_index
                ]
            ),
            "between_group_statistic": float(
                observed_between_group[
                    topic_index
                ]
            ),
            "permutation_p_value": float(
                (
                    global_exceed[
                        topic_index
                    ]
                    + 1
                )
                / (
                    PERMUTATION_REPLICATES
                    + 1
                )
            ),
        })

    global_permutation_table = pd.DataFrame(
        global_permutation_rows
    )

    global_permutation_table[
        "p_fdr_bh"
    ] = benjamini_hochberg(
        global_permutation_table[
            "permutation_p_value"
        ].to_numpy()
    )

    cell_permutation_rows = []

    for type_index, deposit_type in enumerate(
        TYPE_ORDER
    ):
        for topic_index in range(K):
            cell_permutation_rows.append({
                "cohort": (
                    cohort_name
                ),
                "deposit_type": (
                    deposit_type
                ),
                "topic": (
                    topic_index + 1
                ),
                "topic_label": (
                    TOPIC_LABELS[
                        topic_index
                    ]
                ),
                "observed_mean_deviation": float(
                    observed_deviation[
                        type_index,
                        topic_index,
                    ]
                ),
                "permutation_p_two_sided": float(
                    (
                        cell_exceed[
                            type_index,
                            topic_index,
                        ]
                        + 1
                    )
                    / (
                        PERMUTATION_REPLICATES
                        + 1
                    )
                ),
                "permutation_p_maxT": float(
                    (
                        max_cell_exceed[
                            type_index,
                            topic_index,
                        ]
                        + 1
                    )
                    / (
                        PERMUTATION_REPLICATES
                        + 1
                    )
                ),
            })

    cell_permutation_table = pd.DataFrame(
        cell_permutation_rows
    )

    enrichment_table = (
        enrichment_table.merge(
            cell_permutation_table[
                [
                    "deposit_type",
                    "topic",
                    "observed_mean_deviation",
                    "permutation_p_two_sided",
                    "permutation_p_maxT",
                ]
            ],
            on=[
                "deposit_type",
                "topic",
            ],
            how="left",
            validate="one_to_one",
        )
    )

    enrichment_table.to_csv(
        TABLE_DIR
        / f"{cohort_name}__enrichment_bootstrap_and_permutation.csv",
        index=False,
    )

    global_permutation_table.to_csv(
        TABLE_DIR
        / f"{cohort_name}__global_topic_permutation_tests.csv",
        index=False,
    )

    # ---------------------------------------------------------
    # Dominant-attribution contingency
    # ---------------------------------------------------------

    contingency = pd.crosstab(
        pd.Categorical(
            labels,
            categories=TYPE_ORDER,
            ordered=True,
        ),
        pd.Categorical(
            dominant_values + 1,
            categories=list(
                range(
                    1,
                    K + 1,
                )
            ),
            ordered=True,
        ),
        dropna=False,
    )

    chi_square, chi_p, degrees_freedom, expected = (
        chi2_contingency(
            contingency.to_numpy(),
            correction=False,
        )
    )

    residuals = (
        adjusted_standardized_residuals(
            contingency.to_numpy()
        )
    )

    residuals_df = pd.DataFrame(
        residuals,
        index=TYPE_ORDER,
        columns=topic_columns,
    )

    contingency.to_csv(
        TABLE_DIR
        / f"{cohort_name}__dominant_topic_contingency.csv"
    )

    residuals_df.to_csv(
        TABLE_DIR
        / f"{cohort_name}__dominant_topic_adjusted_residuals.csv"
    )

    contingency_summary = pd.DataFrame([
        {
            "cohort": cohort_name,
            "n": int(n),
            "chi_square": float(
                chi_square
            ),
            "degrees_freedom": int(
                degrees_freedom
            ),
            "p_value": float(
                chi_p
            ),
            "cramers_v": (
                cramers_v(
                    contingency.to_numpy()
                )
            ),
        }
    ])

    contingency_summary.to_csv(
        TABLE_DIR
        / f"{cohort_name}__dominant_topic_contingency_summary.csv",
        index=False,
    )

    # ---------------------------------------------------------
    # Pre-specified label checks
    # ---------------------------------------------------------

    planned_rows = []

    for topic_number in range(
        1,
        K + 1,
    ):
        expected_type = (
            EXPECTED_TYPE_BY_TOPIC.get(
                topic_number
            )
        )

        topic_enrichment = (
            enrichment_table[
                enrichment_table[
                    "topic"
                ].eq(
                    topic_number
                )
            ]
            .sort_values(
                "enrichment_ratio",
                ascending=False,
            )
        )

        strongest_row = (
            topic_enrichment.iloc[0]
        )

        if expected_type is None:
            status = (
                "cross_type_or_overprint_mode"
            )

            planned_rows.append({
                "cohort": cohort_name,
                "topic": topic_number,
                "topic_label": (
                    TOPIC_LABELS[
                        topic_number - 1
                    ]
                ),
                "pre_specified_expected_type": (
                    None
                ),
                "strongest_observed_type": (
                    strongest_row[
                        "deposit_type"
                    ]
                ),
                "strongest_enrichment_ratio": float(
                    strongest_row[
                        "enrichment_ratio"
                    ]
                ),
                "ratio_ci_low": float(
                    strongest_row[
                        "ratio_ci_low"
                    ]
                ),
                "ratio_ci_high": float(
                    strongest_row[
                        "ratio_ci_high"
                    ]
                ),
                "permutation_p_maxT": float(
                    strongest_row[
                        "permutation_p_maxT"
                    ]
                ),
                "validation_status": status,
            })

            continue

        expected_row = (
            topic_enrichment[
                topic_enrichment[
                    "deposit_type"
                ].eq(
                    expected_type
                )
            ].iloc[0]
        )

        ratio = float(
            expected_row[
                "enrichment_ratio"
            ]
        )

        ci_low = float(
            expected_row[
                "ratio_ci_low"
            ]
        )

        p_max = float(
            expected_row[
                "permutation_p_maxT"
            ]
        )

        if (
            ratio > 1
            and ci_low > 1
            and p_max < 0.05
        ):
            status = (
                "supported"
            )
        elif ratio > 1:
            status = (
                "partially_supported"
            )
        else:
            status = (
                "not_supported"
            )

        planned_rows.append({
            "cohort": cohort_name,
            "topic": topic_number,
            "topic_label": (
                TOPIC_LABELS[
                    topic_number - 1
                ]
            ),
            "pre_specified_expected_type": (
                expected_type
            ),
            "strongest_observed_type": (
                strongest_row[
                    "deposit_type"
                ]
            ),
            "strongest_enrichment_ratio": float(
                strongest_row[
                    "enrichment_ratio"
                ]
            ),
            "expected_type_enrichment_ratio": (
                ratio
            ),
            "ratio_ci_low": float(
                expected_row[
                    "ratio_ci_low"
                ]
            ),
            "ratio_ci_high": float(
                expected_row[
                    "ratio_ci_high"
                ]
            ),
            "permutation_p_maxT": (
                p_max
            ),
            "validation_status": status,
        })

    planned_table = pd.DataFrame(
        planned_rows
    )

    planned_table.to_csv(
        TABLE_DIR
        / f"{cohort_name}__pre_specified_topic_label_validation.csv",
        index=False,
    )

    # ---------------------------------------------------------
    # Figures
    # ---------------------------------------------------------

    def heatmap(
        matrix,
        row_labels,
        column_labels,
        title,
        colourbar_label,
        output_name,
        value_format=".3f",
        annotations=None,
        centre=None,
        figsize=(12, 6),
        x_rotation=35,
        x_ha="right",
        title_visible=True,
    ):
        fig, ax = plt.subplots(
            figsize=figsize
        )

        if centre is None:
            image = ax.imshow(
                matrix,
                aspect="auto",
            )
        else:
            maximum = np.max(
                np.abs(
                    matrix
                    - centre
                )
            )

            image = ax.imshow(
                matrix,
                aspect="auto",
                vmin=(
                    centre
                    - maximum
                ),
                vmax=(
                    centre
                    + maximum
                ),
            )

        ax.set_xticks(
            np.arange(
                len(
                    column_labels
                )
            )
        )

        ax.set_xticklabels(
            column_labels,
            rotation=x_rotation,
            ha=x_ha,
        )

        ax.set_yticks(
            np.arange(
                len(
                    row_labels
                )
            )
        )

        ax.set_yticklabels(
            row_labels
        )

        if title_visible:
            ax.set_title(title)

        for row_index in range(
            matrix.shape[0]
        ):
            for column_index in range(
                matrix.shape[1]
            ):
                text = format(
                    matrix[
                        row_index,
                        column_index,
                    ],
                    value_format,
                )

                if annotations is not None:
                    text += annotations[
                        row_index
                    ][
                        column_index
                    ]

                ax.text(
                    column_index,
                    row_index,
                    text,
                    ha="center",
                    va="center",
                    fontsize=8,
                )

        colourbar = fig.colorbar(
            image,
            ax=ax,
        )

        colourbar.set_label(
            colourbar_label
        )

        fig.tight_layout()

        fig.savefig(
            FIG_DIR
            / output_name,
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)

    mean_attr_row_labels = [
        "Porphyry",
        "VMS",
        "Sediment-hosted",
        "Magmatic sulphide",
        "IOCG",
    ]

    mean_attr_column_labels = [
        f"T{topic}"
        for topic in range(1, K + 1)
    ]

    heatmap(
        mean_r,
        mean_attr_row_labels,
        mean_attr_column_labels,
        "Mean posterior topic attribution by deposit type",
        "Mean posterior attribution",
        (
            f"{cohort_name}__mean_attribution_heatmap.png"
        ),
        figsize=(8.8, 4.8),
        x_rotation=0,
        x_ha="center",
        title_visible=True,
    )

    enrichment_matrix = (
        enrichment_table.pivot(
            index="deposit_type",
            columns="topic",
            values="enrichment_ratio",
        )
        .reindex(
            index=TYPE_ORDER,
            columns=list(
                range(
                    1,
                    K + 1,
                )
            ),
        )
        .to_numpy()
    )

    p_matrix = (
        enrichment_table.pivot(
            index="deposit_type",
            columns="topic",
            values="permutation_p_maxT",
        )
        .reindex(
            index=TYPE_ORDER,
            columns=list(
                range(
                    1,
                    K + 1,
                )
            ),
        )
        .to_numpy()
    )

    significance_annotations = [
        [
            format_significance(
                p_matrix[
                    row_index,
                    column_index,
                ]
            )
            for column_index
            in range(K)
        ]
        for row_index
        in range(
            len(TYPE_ORDER)
        )
    ]

    heatmap(
        enrichment_matrix,
        TYPE_ORDER,
        TOPIC_LABELS,
        (
            f"Topic enrichment relative to cohort mean — "
            f"{cohort_name}"
        ),
        "Enrichment ratio",
        (
            f"{cohort_name}__enrichment_ratio_heatmap.png"
        ),
        value_format=".2f",
        annotations=(
            significance_annotations
        ),
        centre=1.0,
    )

    heatmap(
        residuals,
        TYPE_ORDER,
        TOPIC_LABELS,
        (
            f"Adjusted residuals of dominant attribution — "
            f"{cohort_name}"
        ),
        "Adjusted standardized residual",
        (
            f"{cohort_name}__dominant_topic_residual_heatmap.png"
        ),
        value_format=".2f",
        centre=0.0,
    )

    # Separate violin plot for each topic.
    for topic_index in range(K):
        fig, ax = plt.subplots(
            figsize=(9, 5)
        )

        data = [
            r_values[
                type_indices[
                    deposit_type
                ],
                topic_index,
            ]
            for deposit_type
            in TYPE_ORDER
        ]

        violin = ax.violinplot(
            data,
            showmeans=True,
            showmedians=True,
            showextrema=False,
        )

        ax.set_xticks(
            np.arange(
                1,
                len(TYPE_ORDER) + 1,
            )
        )

        ax.set_xticklabels(
            TYPE_ORDER,
            rotation=25,
            ha="right",
        )

        ax.set_ylabel(
            "Posterior attribution"
        )

        ax.set_title(
            f"{TOPIC_LABELS[topic_index]} — {cohort_name}"
        )

        fig.tight_layout()

        fig.savefig(
            FIG_DIR
            / (
                f"{cohort_name}__T{topic_index+1}"
                "__attribution_violin.png"
            ),
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)

    return {
        "mean_r": mean_r_df,
        "median_r": median_r_df,
        "mean_theta": mean_theta_df,
        "median_theta": median_theta_df,
        "kruskal": kw_table,
        "pairwise": pairwise_table,
        "enrichment": enrichment_table,
        "global_permutation": (
            global_permutation_table
        ),
        "contingency": contingency,
        "residuals": residuals_df,
        "contingency_summary": (
            contingency_summary
        ),
        "planned": planned_table,
    }


## 7. Run the validation on both cohorts

In [ ]:
results = {}

for cohort_name, mask in cohorts.items():
    print("\n" + "=" * 90)
    print("Analysing:", cohort_name)
    print(
        "Deposits:",
        int(mask.sum()),
    )
    print("=" * 90)

    results[
        cohort_name
    ] = analyse_cohort(
        cohort_name,
        mask,
    )

    print("\nPre-specified label checks:")
    print(
        results[
            cohort_name
        ][
            "planned"
        ][
            [
                "topic",
                "topic_label",
                "pre_specified_expected_type",
                "strongest_observed_type",
                "expected_type_enrichment_ratio",
                "ratio_ci_low",
                "ratio_ci_high",
                "permutation_p_maxT",
                "validation_status",
            ]
        ].to_string(
            index=False
        )
    )


## 8. Compare the complete and coordinate-valid cohorts

In [ ]:
full_mean = (
    results[
        "full_1335"
    ][
        "mean_r"
    ].to_numpy()
)

valid_mean = (
    results[
        "coordinate_valid_1237"
    ][
        "mean_r"
    ].to_numpy()
)

matrix_correlation = float(
    np.corrcoef(
        full_mean.ravel(),
        valid_mean.ravel(),
    )[0, 1]
)

maximum_absolute_difference = float(
    np.max(
        np.abs(
            full_mean
            - valid_mean
        )
    )
)

full_enrichment = (
    results[
        "full_1335"
    ][
        "enrichment"
    ][
        [
            "deposit_type",
            "topic",
            "enrichment_ratio",
            "permutation_p_maxT",
        ]
    ]
    .rename(
        columns={
            "enrichment_ratio": (
                "enrichment_ratio_full"
            ),
            "permutation_p_maxT": (
                "p_maxT_full"
            ),
        }
    )
)

valid_enrichment = (
    results[
        "coordinate_valid_1237"
    ][
        "enrichment"
    ][
        [
            "deposit_type",
            "topic",
            "enrichment_ratio",
            "permutation_p_maxT",
        ]
    ]
    .rename(
        columns={
            "enrichment_ratio": (
                "enrichment_ratio_valid"
            ),
            "permutation_p_maxT": (
                "p_maxT_valid"
            ),
        }
    )
)

cohort_enrichment_comparison = (
    full_enrichment.merge(
        valid_enrichment,
        on=[
            "deposit_type",
            "topic",
        ],
        how="inner",
        validate="one_to_one",
    )
)

cohort_enrichment_comparison[
    "same_enrichment_direction"
] = (
    (
        cohort_enrichment_comparison[
            "enrichment_ratio_full"
        ]
        > 1
    )
    == (
        cohort_enrichment_comparison[
            "enrichment_ratio_valid"
        ]
        > 1
    )
)

cohort_enrichment_comparison.to_csv(
    TABLE_DIR
    / "full_vs_coordinate_valid_enrichment_comparison.csv",
    index=False,
)

cohort_comparison_summary = pd.DataFrame([
    {
        "mean_attribution_matrix_correlation": (
            matrix_correlation
        ),
        "maximum_absolute_mean_difference": (
            maximum_absolute_difference
        ),
        "enrichment_direction_agreement_fraction": float(
            cohort_enrichment_comparison[
                "same_enrichment_direction"
            ].mean()
        ),
        "full_dominant_contingency_cramers_v": float(
            results[
                "full_1335"
            ][
                "contingency_summary"
            ][
                "cramers_v"
            ].iloc[0]
        ),
        "valid_dominant_contingency_cramers_v": float(
            results[
                "coordinate_valid_1237"
            ][
                "contingency_summary"
            ][
                "cramers_v"
            ].iloc[0]
        ),
    }
])

cohort_comparison_summary.to_csv(
    TABLE_DIR
    / "full_vs_coordinate_valid_summary.csv",
    index=False,
)

print(
    cohort_comparison_summary.round(
        5
    ).to_string(
        index=False
    )
)


## 9. Reviewer-oriented summary

In [ ]:
reviewer_rows = []

for cohort_name in [
    "full_1335",
    "coordinate_valid_1237",
]:
    kw_table = results[
        cohort_name
    ][
        "kruskal"
    ]

    planned_table = results[
        cohort_name
    ][
        "planned"
    ]

    contingency_summary = results[
        cohort_name
    ][
        "contingency_summary"
    ].iloc[0]

    reviewer_rows.append({
        "cohort": cohort_name,
        "n": int(
            cohorts[
                cohort_name
            ].sum()
        ),
        "topics_with_fdr_significant_type_heterogeneity": int(
            (
                kw_table[
                    "p_fdr_bh"
                ]
                < 0.05
            ).sum()
        ),
        "median_epsilon_squared": float(
            kw_table[
                "epsilon_squared"
            ].median()
        ),
        "pre_specified_type_specific_topics_supported": int(
            (
                planned_table[
                    "validation_status"
                ]
                == "supported"
            ).sum()
        ),
        "pre_specified_type_specific_topics_partially_supported": int(
            (
                planned_table[
                    "validation_status"
                ]
                == "partially_supported"
            ).sum()
        ),
        "pre_specified_type_specific_topics_not_supported": int(
            (
                planned_table[
                    "validation_status"
                ]
                == "not_supported"
            ).sum()
        ),
        "dominant_attribution_chi_square_p": float(
            contingency_summary[
                "p_value"
            ]
        ),
        "dominant_attribution_cramers_v": float(
            contingency_summary[
                "cramers_v"
            ]
        ),
    })

reviewer_summary = pd.DataFrame(
    reviewer_rows
)

reviewer_summary.to_csv(
    TABLE_DIR
    / "reviewer_topic_validation_summary.csv",
    index=False,
)

print(
    reviewer_summary.round(
        5
    ).to_string(
        index=False
    )
)

print(
    "\nInterpretation rule: "
    "type-specific enrichment supports an association label, "
    "not a one-to-one genetic classification. "
    "T2, T3, and T7 are intentionally evaluated as cross-type "
    "overprint/background/residual modes."
)


## 10. Save provenance and create the output archive

In [ ]:
manifest = {
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "analysis": (
        "post_hoc_topic_validation_against_known_deposit_types"
    ),
    "final_model": {
        "name": (
            "HGCTM-S / M7b"
        ),
        "likelihood": (
            "Dirichlet-Multinomial"
        ),
        "K": int(K),
        "F": int(F),
        "cohort_n": int(
            EXPECTED_N
        ),
        "warm_start": (
            "LDA"
        ),
        "sparsity_schedule": (
            "annealed family gate"
        ),
    },
    "inputs": {
        "final_results_folder": str(
            FINAL_RESULTS
        ),
        "stage0_folder": str(
            STAGE0
        ),
    },
    "cohorts": {
        "full_1335": int(
            cohorts[
                "full_1335"
            ].sum()
        ),
        "coordinate_valid_1237": int(
            cohorts[
                "coordinate_valid_1237"
            ].sum()
        ),
    },
    "validation_quantities": [
        "posterior attribution weights",
        "topic prevalence theta",
        "dominant posterior attribution",
    ],
    "tests": [
        "Kruskal-Wallis",
        "epsilon-squared",
        "pairwise Mann-Whitney with Holm correction",
        "stratified bootstrap enrichment ratios",
        "label permutation with max-statistic adjustment",
        "chi-square and Cramer's V",
    ],
    "bootstrap_replicates": int(
        BOOTSTRAP_REPLICATES
    ),
    "permutation_replicates": int(
        PERMUTATION_REPLICATES
    ),
    "important_interpretation": (
        "Deposit-type labels were not used to fit HGCTM-S. "
        "The analysis validates statistical enrichment and geological "
        "association of assemblage modes; it does not treat topics as "
        "one-to-one genetic deposit classes."
    ),
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

readme = '''
HGCTM post hoc topic validation against known deposit types

Primary quantity:
- posterior attribution weights combining theta, context-conditioned
  topic content, and observed mineral-family counts.

Secondary quantities:
- theta prevalence;
- dominant posterior attribution.

Cohorts:
- all 1,335 final-model deposits;
- 1,237 coordinate-valid deposits.

No classifier is trained and deposit-type labels are not used for
model fitting. Significant enrichment supports geological association
labels but not one-to-one genetic classification.
'''.strip()

(
    OUT / "README.txt"
).write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(
        WORK
        / "HGCTM_Topic_Validation_Against_Deposit_Types_K7_Outputs"
    ),
    "zip",
    root_dir=OUT,
)

print("Output archive:")
print(archive_path)

print("\nKey tables:")
for path in sorted(
    TABLE_DIR.iterdir()
):
    print(" -", path.name)

print("\nFigures:")
for path in sorted(
    FIG_DIR.iterdir()
):
    print(" -", path.name)
